In [ ]:
!pip install -q groq google-generativeai pydantic


# 1. Drive Mount & Setup
Run this cell to connect Google Drive and configure the input/output paths.


In [ ]:
from google.colab import drive
import os
import shutil
import json

# Mount your Google Drive
drive.mount('/content/drive')

# --- CONFIGURATION PATHS ---
# Your own Google Drive folder for this project; set CASERAG_DRIVE_DIR to point to it.
# It should contain ilpcsr_dataset/il-pcsr_dataset.zip. Summaries are written to Completed_Summaries/.
drive_dir = os.environ.get('CASERAG_DRIVE_DIR', '/content/drive/MyDrive/<your-folder>')
drive_zip_path = os.path.join(drive_dir, 'ilpcsr_dataset', 'il-pcsr_dataset.zip')
local_zip_path = '/content/il-pcsr_dataset.zip'
local_extract_dir = '/content/dataset'
drive_output_dir = os.path.join(drive_dir, 'Completed_Summaries')

# Ensure the output directory exists
os.makedirs(drive_output_dir, exist_ok=True)


# 2. Fast Local Extraction (The "Jugaad")
Unzips the dataset into Colab's fast local SSD storage to bypass Google Drive's slow reading speeds.


In [ ]:
# We only extract if the directory doesn't exist yet to save time
if not os.path.exists(local_extract_dir):
    print("Copying zip from Drive to fast local storage...")
    shutil.copy2(drive_zip_path, local_zip_path)
    
    print("Extracting zip locally...")
    import zipfile
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_extract_dir)
        
    print("Extraction complete!")
else:
    print("Local dataset already exists. Skipping extraction.")

# Note: Adjust 'il-pcsr_dataset' and 'corpus' if your zip structure is slightly different.
DATASET_DIR = os.path.join(local_extract_dir, "il-pcsr_dataset", "corpus")
all_files = sorted([f for f in os.listdir(DATASET_DIR) if f.endswith('.json')])
print(f"Total files found in dataset: {len(all_files)}")


# 3. Colab Secrets Setup & Model Configuration
Ensure you have added `GROQ_API_KEY` (or `GOOGLE_API_KEY`) to the Colab Secrets tab!


In [ ]:
from groq import Groq
import google.generativeai as genai
from pydantic import ValidationError, BaseModel, Field
from typing import List, Dict
from google.colab import userdata

# Get the API Key from Colab Secrets (the key icon on the left)
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except Exception:
    pass # Try getting google api key if groq fails

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except Exception:
    pass

TEXT_KEYS = [
    'Fact',
    'Issue',
    "Petitioner's Argument",
    "Respondent's Argument",
    'Analysis of the law',
    'Precedent Analysis',
    "Court's Reasoning",
    'Conclusion',
    'Court Disclosure',
    'Uncategorized'
]

GROQ_MODELS = ["openai/gpt-oss-120b", "meta-llama/llama-4-scout-17b-16e-instruct"]
GOOGLE_MODELS = ["gemini-2.5-pro", "gemini-2.5-flash"]

class LLMCaseSummary(BaseModel):
    """A structured summary of the core legal arguments and decisions."""
    key_facts: str = Field(..., description="A concise, one-to-two-sentence summary of the essential facts of the case.")
    legal_issues: List[str] = Field(..., description="List as many core and subsidiary legal questions as possible that the court had to address in this case.")
    holding: str = Field(..., description="The court's final decision or ruling, stated clearly in one or two sentences.")

def _call_groq_api(messages: List[Dict], model_name: str) -> str:
    print(f"--- Calling Groq API with model: {model_name} ---")
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model_name,
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    return chat_completion.choices[0].message.content

def _call_google_api(messages: List[Dict], model_name: str) -> str:
    print(f"--- Calling Google API with model: {model_name} ---")
    genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
    
    system_prompt = ""
    user_prompts = []
    for msg in messages:
        if msg["role"] == "system":
            system_prompt = msg["content"]
        else:
            user_prompts.append(msg["content"])
    
    model = genai.GenerativeModel(
        model_name,
        system_instruction=system_prompt
    )
    
    response = model.generate_content(
        "\n".join(user_prompts),
        generation_config={"response_mime_type": "application/json"}
    )
    
    return response.text.strip().replace("```json", "").replace("```", "")

def generate_structured_summary(case_text: str, model_name: str = "openai/gpt-oss-120b") -> LLMCaseSummary | None:
    system_prompt = (
        "You are a specialized legal AI assistant. Your sole purpose is to read a court "
        "judgment and generate a structured summary that perfectly matches the provided JSON schema. "
        "You MUST adhere to the schema. Do not extract other facts or create your own keys."
    )
    
    user_prompt = f"""
    You MUST generate a JSON object that conforms exactly to the following structure.
    Do not add, omit, or change any keys. Follow the example provided.

    ### EXAMPLE ###
    Input Text: "The appellant, John Doe, was convicted under Section 307 for attempted murder after an altercation. The key issue was whether the act constituted a 'heinous crime' for bail purposes. The court referenced the landmark case of 'State v. Smith' which defined heinous crimes. The appeal was ultimately dismissed."
    
    Output JSON:
    {{
        "key_facts": "The appellant was convicted for attempted murder following an altercation.",
        "legal_issues": ["Whether the act constituted a 'heinous crime' for the purpose of denying bail."],
        "holding": "The court dismissed the appeal, implying the act was considered heinous."
    }}
    ### END OF EXAMPLE ###

    Now, analyze the following court case text and generate the corresponding JSON object.

    ### CASE TEXT ###
    {case_text}
    ### END OF CASE TEXT ###
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    json_response = ""
    try:
        if model_name in GROQ_MODELS:
            json_response = _call_groq_api(messages, model_name)
        elif model_name in GOOGLE_MODELS:
            json_response = _call_google_api(messages, model_name)
        else:
            print(f"Error: Model '{model_name}' is not configured with any provider.")
            return None
            
        validated_summary = LLMCaseSummary.model_validate_json(json_response)
        return validated_summary

    except ValidationError as e:
        print(f"--- Pydantic Validation Error ---\n{e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


# 4. Workload Slicing Configuration
Change `START_INDEX` and `END_INDEX` when running multiple notebooks to prevent overlap.


In [ ]:
# 4. Workload Slicing Configuration
# You only want files where doc_id % 4 == 3
# We then split those files into 3 parts for your 3 notebooks.

NOTEBOOK_ID = 0  # Change this to 0, 1, or 2 for your 3 notebooks
TOTAL_NOTEBOOKS = 3

my_assigned_files = []
for f in all_files:
    # Check if the file is already completed in the target Drive folder
    if os.path.exists(os.path.join(drive_output_dir, f)):
        continue
        
    # Extract the doc_id from the filename (e.g., '1000149.json' -> 1000149)
    try:
        doc_id = int(f.replace('.json', ''))
    except ValueError:
        continue  # Skip files that aren't just numbers
        
    # 1. Global Rule: Only process if doc_id % 4 == 3
    if doc_id % 4 == 3:
        # 2. Distributed Rule: Split the remaining files between the notebooks
        if doc_id % TOTAL_NOTEBOOKS == NOTEBOOK_ID:
            my_assigned_files.append(f)

print(f"This notebook (ID {NOTEBOOK_ID}) is assigned to process {len(my_assigned_files)} REMAINING files.")


# 5. Distributed Execution Loop & Auto-Save
Runs generation and saves directly to the `Completed_Summaries` folder in Google Drive.


In [ ]:
# 5. Distributed Execution Loop & Auto-Save
MODEL_TO_USE = "meta-llama/llama-4-scout-17b-16e-instruct"

print(f"--- Starting Distributed Generation ---")
print(f"Target Output Directory: {drive_output_dir}")

for i, filename in enumerate(my_assigned_files):
    output_filepath = os.path.join(drive_output_dir, filename)
    
    # 1. Check Google Drive Output Folder
    if os.path.exists(output_filepath):
        print(f"[{i+1}/{len(my_assigned_files)}] {filename} already completed on Drive. Skipping.")
        continue
        
    print(f"\n[{i+1}/{len(my_assigned_files)}] Processing file: {filename}")
    
    # 2. Read from Fast Local Storage
    local_filepath = os.path.join(DATASET_DIR, filename)
    try:
        with open(local_filepath, 'r', encoding='utf-8') as f:
            case_json = json.load(f)
            
        summary_data = case_json.get("Summary")
        if isinstance(summary_data, dict) and summary_data:
            # File on local disk has summary? Copy to drive directly.
            print("Summary already in local file. Copying to Drive.")
            with open(output_filepath, 'w', encoding='utf-8') as f:
                json.dump(case_json, f, indent=4, ensure_ascii=False)
            continue
            
        # 3. Dynamic Text Aggregation
        full_text_parts = []
        for key in TEXT_KEYS:
            if key in case_json and isinstance(case_json[key], list):
                full_text_parts.extend(case_json[key])
                
        full_text = " ".join(full_text_parts)

        if not full_text.strip():
            print("No content found to summarize. Skipping.")
            continue
            
        # 4. Generate LLM Summary
        llm_summary = generate_structured_summary(full_text, model_name=MODEL_TO_USE)
        
        if llm_summary:
            final_summary_dict = llm_summary.model_dump()
            final_summary_dict['key_statutes_cited'] = case_json.get('relevant_statutes') or []
            final_summary_dict['key_precedents_cited'] = case_json.get('relevant_precedents') or []
            
            case_json['Summary'] = final_summary_dict
            
            # 5. Save DIRECTLY to Google Drive
            with open(output_filepath, 'w', encoding='utf-8') as f:
                json.dump(case_json, f, indent=4, ensure_ascii=False)
                
            print(f"Successfully generated summary and auto-saved to Drive: {filename}")
            
    except Exception as e:
        print(f"An error occurred while processing {filename}: {e}")
